In [53]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [54]:
from enums.Dataset import Dataset
from utils import utils
from enums.Participant import NurseParticipant

dataset = Dataset.NURSE
with_features = True
participant = NurseParticipant.n_F5

df = utils.load_data(dataset=dataset, with_features=with_features, which="all")

In [55]:
from enums.Model import Model
from enums.ResamplingMethod import ResamplingMethod
from enums.ScalingMethod import ScalingMethod

x_train, x_val, x_test, y_train, y_val, y_test = utils.split_data(
    df=df, with_features=with_features, model=Model.SHALLOW_NN
)

scaler = utils.get_scaler(method=ScalingMethod.STANDARDSCALER)
resampler = utils.get_resampler(method=ResamplingMethod.UNDERSAMPLING)

In [56]:
from model.ShallowNNModel import ShallowNNModel

if scaler is not None:
    _ = scaler.fit_transform(x_train)
    x_val = scaler.transform(x_val)
model = ShallowNNModel(
    scaler=scaler,
    resampler=resampler,
    input_shape=x_train.shape[1],
    dataset=dataset,
    with_features=with_features,
    val_data=(x_val, y_val),
)

In [57]:
model.fit(x_train=x_train, y_train=y_train, run_info=None)

Epoch 1/50
604/604 - 4s - 6ms/step - accuracy: 0.7718 - loss: 0.4679 - val_accuracy: 0.6576 - val_loss: 2.0507
Epoch 2/50
604/604 - 3s - 6ms/step - accuracy: 0.8855 - loss: 0.2788 - val_accuracy: 0.6606 - val_loss: 3.2299
Epoch 3/50
604/604 - 3s - 5ms/step - accuracy: 0.9327 - loss: 0.1822 - val_accuracy: 0.6880 - val_loss: 4.1467
Epoch 4/50
604/604 - 3s - 5ms/step - accuracy: 0.9569 - loss: 0.1255 - val_accuracy: 0.6993 - val_loss: 4.9303
Epoch 5/50
604/604 - 3s - 6ms/step - accuracy: 0.9713 - loss: 0.0929 - val_accuracy: 0.6950 - val_loss: 5.1991
Epoch 6/50
604/604 - 3s - 6ms/step - accuracy: 0.9791 - loss: 0.0685 - val_accuracy: 0.7090 - val_loss: 6.5551


In [58]:
pred_train = model.predict(x=x_train)
scores_train, _ = model.evaluate(pred=pred_train, y_true=y_train)
pred_test = model.predict(x=x_test)
scores_test, _ = model.evaluate(pred=pred_test, y_true=y_test)

print(f"Train scores: {scores_train}")
print(f"Test scores: {scores_test}")

1412/1412 - 2s - 1ms/step
432/432 - 1s - 1ms/step
Train scores: {'accuracy': 0.9733673599309135, 'recall': 0.9934258320206576, 'precision': 0.9725586570149002, 'f1': 0.9828815012756146, 'confusion_matrix': {'tp': 138115, 'tn': 37717, 'fp': 914, 'fn': 3897}}
Test scores: {'accuracy': 0.6854398089517676, 'recall': 0.7853616608630494, 'precision': 0.823302451308828, 'f1': 0.8038846340390494, 'confusion_matrix': {'tp': 35635, 'tn': 2252, 'fp': 9739, 'fn': 7648}}
